In [ ]:
from atlite.gis import ExclusionContainer, shape_availability, padded_transform_and_shape
import time
import geopandas as gpd
import pandas as pd
import numpy as np
import rasterio as rio
from rasterio.enums import Resampling
from rasterio.features import rasterize
from numpy import empty
from scipy.ndimage import binary_dilation as dilation
from shapely.geometry import Polygon, box
import shapely

from workflow.scripts.extra_functions import get_bounding

In [ ]:
## method
method = snakemake.params.method # atlite

# input data
europe_shp = snakemake.input.europe_shapefile

# parameters
aggregated_regions = snakemake.params.aggregated_regions
polygon_borders = snakemake.params.polygon_borders
method = snakemake.params.method
target_crs = snakemake.params.target_crs # target EPSG

# output
output_path = snakemake.output.europe_raster

In [ ]:
# resolution
res = 100

# borders
[rectx1, rectx2, recty1, recty2] = polygon_borders
polygon_bounds = get_bounding(target_crs,rectx1,rectx2,recty1,recty2)

In [ ]:
t=time.time()


europe = (
    gpd
    .read_file(europe_shp)
    .replace({"EL": "GR"})
    .query("NUTS_ID == @aggregated_regions")
    .set_index(["NUTS_ID"])
    .loc[:,['geometry']]
    .to_crs("EPSG:4326")
)


polygon = Polygon(
[
    (rectx1, recty1),
    (rectx1, recty2),
    (rectx2, recty2),
    (rectx2, recty1),
    (rectx1, recty1),
]
)

polygon = shapely.segmentize(polygon, max_segment_length=0.5)

europe = (
    gpd
    .clip(europe, polygon)
)
# Remove Jan Mayen
nor = europe.query("NUTS_ID =='NO'").clip(box(0, recty1, rectx2, recty2))
europe = pd.concat([europe.query("NUTS_ID != 'NO'"), nor])
europe = europe.to_crs(target_crs)

onshore = (
    europe
    .assign(area = 1)
    .dissolve(by='area')
    .reset_index()
)
onshore["area"] = onshore["area"].astype("uint8")

3.893890380859375


In [ ]:
t=time.time()

if method=="manual":
    dst_transform, dst_shape = padded_transform_and_shape(
            bounds=rio.features.bounds(polygon_bounds), # cutout bounds
            res=res # resolution
        )
    
    raster_onshore_original = empty(dst_shape)
    raster_onshore_original = rio.features.rasterize(
        zip(onshore.geometry, onshore["area"].values),
        out_shape=dst_shape,
        transform=dst_transform,
        fill=0,
        all_touched=True,
        dtype="uint8"
    )
    
    meta = {
        'driver': 'GTiff',
        'dtype': 'uint8',
        'nodata': None,
        'width': raster_onshore_original.shape[1],
        'height': raster_onshore_original.shape[0],
        'count': 1,
        'crs': target_crs,
        'transform': dst_transform,
        'compress': 'lzw'
    }
    
    # Save to file
    with rio.open(output_path, 'w', **meta) as dst:
        dst.write(raster_onshore_original, 1)
    
    print(time.time()-t)

16.001119136810303


In [ ]:
t=time.time()
if method=="atlite":

    exclusion_container = ExclusionContainer(crs=target_crs)
    exclusion_container.add_geometry(europe)
    masked, transform = shape_availability(polygon_bounds.geometry, exclusion_container)
    raster_reprojected = (~masked).astype("uint8")
    
    meta = {
        'driver': 'GTiff',
        'dtype': 'uint8',
        'nodata': None,
        'width': raster_reprojected.shape[1],
        'height': raster_reprojected.shape[0],
        'count': 1,
        'crs': target_crs,
        'transform': transform,
        'compress': 'lzw'
    }
    
    # Save to file
    with rio.open(output_path, 'w', **meta) as dst:
        dst.write(raster_reprojected, 1)
    
    print(time.time()-t)

330.19193291664124


In [ ]:
if method=="test":
    f = open(output_path, "x")